# Module 8. GRPO 실습: PPO와 무엇이 같고 무엇이 다른가

이 노트북의 목표는 다음 5가지입니다.

1. `module3_grpo_dataset.jsonl`을 이용해 **GRPOTrainer** 기반 GRPO 실습을 수행한다.
2. 초기 정책은 **Module 4의 SFT 결과**를 사용해 PPO/GRPO 비교의 출발선을 맞춘다.
3. PPO와 **같은 reward 철학**(math / format / persona / safety)을 사용한다.
4. **SFT vs GRPO** 출력과 간단 scorecard를 비교한다.
5. 가능하면 Module 7 PPO 결과와 함께 **학습 안정성, 메모리, 구현 난이도, 응답 품질**을 비교 기록한다.

## 핵심 개념

- **DPO**: offline preference alignment
- **PPO**: reward/value 기반 online RL
- **GRPO**: PPO 계열이지만, 그룹 상대 비교 기반 advantage를 사용하고 critic/value-model 의존성을 줄인 온라인 RL 변형

DeepSeekMath는 GRPO를 **PPO의 변형**으로 소개하며, 수학 추론을 강화하면서도 **PPO의 메모리 사용을 줄이는 방향**을 강조합니다. TRL 문서도 GRPO를 online learning algorithm으로 설명하고, prompt마다 여러 completion을 생성해 reward를 계산한 뒤 그룹 상대 비교로 학습한다고 안내합니다.

## Step 1. 패키지 설치

이번 실습에서 사용할 핵심 라이브러리를 설치합니다.

- `transformers`
- `datasets`
- `trl`
- `peft`
- `accelerate`
- `sentencepiece`

> Colab GPU를 사용하는 것을 권장합니다.

In [ ]:
!pip -q install -U transformers datasets trl peft accelerate sentencepiece

## Step 2. 기본 import

In [ ]:
import os
import re
import json
import time
import math
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import AutoPeftModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

## Step 3. 경로 및 기본 설정

기본값은 다음과 같습니다.

- SFT 초기 모델: `/content/module4_sft_output`
- GRPO 데이터셋: `/content/module3_grpo_dataset.jsonl`
- 출력 폴더: `/content/module8_grpo_output`

만약 파일이 없으면 아래 셀에서 경로를 바꾸거나 파일을 업로드해서 사용할 수 있습니다.

In [ ]:
set_seed(42)

SFT_MODEL_DIR = "/content/module4_sft_output"
GRPO_DATA_PATH = "/content/module3_grpo_dataset.jsonl"
OUTPUT_DIR = "/content/module8_grpo_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("SFT_MODEL_DIR:", SFT_MODEL_DIR)
print("GRPO_DATA_PATH:", GRPO_DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

## Step 4. 파일 확인 또는 업로드

이 셀은 이전 모듈 결과가 현재 Colab 세션에 없는 경우를 대비한 fallback입니다.

- `module3_grpo_dataset.jsonl` 이 없으면 업로드할 수 있습니다.
- `module4_sft_output` 이 없으면 기본 모델 fallback을 사용합니다.

> 주의: SFT adapter 폴더 전체를 업로드하는 것은 번거로울 수 있으므로, 가장 좋은 방법은 Module 4 notebook 직후 같은 세션에서 이어서 실행하는 것입니다.

In [ ]:
from google.colab import files

if not os.path.exists(GRPO_DATA_PATH):
    print("GRPO dataset not found. Please upload module3_grpo_dataset.jsonl")
    uploaded = files.upload()
    for name in uploaded.keys():
        if name.endswith(".jsonl"):
            GRPO_DATA_PATH = f"/content/{name}"
            break

print("Using GRPO_DATA_PATH:", GRPO_DATA_PATH)

if not os.path.exists(SFT_MODEL_DIR):
    print("SFT model directory not found. We'll fall back to a base instruct model later.")

## Step 5. 데이터셋 로드

GRPO 실습에서는 **`prompt` 컬럼이 필수**입니다.  
이번 데이터셋은 Module 3에서 만든 것으로, `task_type`, `ground_truth`, `required_keys`, `must_include`, `max_chars`, `must_refuse` 같은 추가 컬럼이 reward 함수에 활용될 수 있습니다.

In [ ]:
dataset = load_dataset("json", data_files=GRPO_DATA_PATH, split="train")
print(dataset)
print(dataset[0])

## Step 6. train / eval 분리 및 소규모 실습 설정

Colab에서 빠르게 실습하기 위해 데이터가 많더라도 일부만 사용하도록 기본 설정합니다.

In [ ]:
# 빠른 실습을 위해 최대 32개만 사용
MAX_SAMPLES = min(32, len(dataset))
small_dataset = dataset.select(range(MAX_SAMPLES))

split_dataset = small_dataset.train_test_split(test_size=0.25, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("train size:", len(train_dataset))
print("eval size :", len(eval_dataset))
print(train_dataset[0])

## Step 7. tokenizer와 초기 모델 로드

우선순위는 다음과 같습니다.

1. Module 4의 SFT adapter가 있으면 그것을 로드
2. 없으면 base instruct 모델 fallback 사용

GRPO는 PPO와 달리 value model 없이 진행하지만, 초기 정책은 여전히 중요합니다.  
이번 실습은 PPO와 최대한 같은 출발선을 맞추기 위해 **SFT 결과를 우선 사용**합니다.

In [ ]:
BASE_FALLBACK_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

if os.path.exists(os.path.join(SFT_MODEL_DIR, "adapter_config.json")):
    print("Loading SFT adapter model from:", SFT_MODEL_DIR)
    model = AutoPeftModelForCausalLM.from_pretrained(
        SFT_MODEL_DIR,
        is_trainable=True,
        torch_dtype=dtype,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR)
    INITIAL_MODEL_NAME = SFT_MODEL_DIR
else:
    print("SFT adapter not found. Loading fallback base model:", BASE_FALLBACK_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_FALLBACK_MODEL,
        torch_dtype=dtype,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_FALLBACK_MODEL)
    INITIAL_MODEL_NAME = BASE_FALLBACK_MODEL

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model.config.use_cache = False

print("INITIAL_MODEL_NAME:", INITIAL_MODEL_NAME)
print("pad_token:", tokenizer.pad_token)
print("padding_side:", tokenizer.padding_side)

## Step 8. baseline generation helper

학습 전/후 출력을 비교하기 위한 생성 함수입니다.

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=96):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## Step 9. reward 함수 정의

이번 reward 함수는 Module 7 PPO와 **같은 철학**으로 설계합니다.

- `math_reward_func`: 정답 일치
- `format_reward_func`: JSON parse + required_keys
- `persona_reward_func`: 공손함 + 길이 제한
- `safety_reward_func`: 안전한 거절

TRL 문서에 따르면 GRPO의 custom reward function은 `prompts`, `completions`, `completion_ids`, 그리고 데이터셋의 추가 컬럼을 입력으로 받을 수 있고, 각 completion에 대한 float reward list를 반환해야 합니다.

In [ ]:
def _first_integer(text):
    m = re.search(r"-?\d+", str(text).replace(",", ""))
    return m.group(0) if m else None

def _safe_json_parse(text):
    try:
        return json.loads(text), True
    except Exception:
        return None, False

def math_reward_func(completions, task_type=None, ground_truth=None, **kwargs):
    rewards = []
    task_type = task_type or [None] * len(completions)
    ground_truth = ground_truth or [None] * len(completions)

    for completion, t, gt in zip(completions, task_type, ground_truth):
        if t != "math":
            rewards.append(None)
            continue
        pred = _first_integer(completion)
        rewards.append(1.0 if pred is not None and str(pred) == str(gt) else 0.0)
    return rewards

def format_reward_func(completions, task_type=None, required_keys=None, **kwargs):
    rewards = []
    task_type = task_type or [None] * len(completions)
    required_keys = required_keys or [None] * len(completions)

    for completion, t, keys in zip(completions, task_type, required_keys):
        if t != "format":
            rewards.append(None)
            continue
        parsed, ok = _safe_json_parse(completion)
        if not ok or not isinstance(parsed, dict):
            rewards.append(0.0)
            continue
        keys = keys or []
        has_all = all(k in parsed for k in keys)
        rewards.append(1.0 if has_all else 0.0)
    return rewards

def persona_reward_func(completions, task_type=None, must_include=None, max_chars=None, **kwargs):
    rewards = []
    task_type = task_type or [None] * len(completions)
    must_include = must_include or [None] * len(completions)
    max_chars = max_chars or [None] * len(completions)

    for completion, t, musts, max_len in zip(completions, task_type, must_include, max_chars):
        if t != "persona":
            rewards.append(None)
            continue

        score = 0.0
        musts = musts or []
        text = str(completion)

        if any(tok in text for tok in musts):
            score += 0.5
        if max_len is not None and len(text) <= int(max_len):
            score += 0.5
        rewards.append(score)
    return rewards

def safety_reward_func(completions, task_type=None, must_refuse=None, **kwargs):
    refusal_markers = ["도와드릴 수", "제공할 수", "안전", "대신", "권장", "도움이 어렵"]

    rewards = []
    task_type = task_type or [None] * len(completions)
    must_refuse = must_refuse or [None] * len(completions)

    for completion, t, must in zip(completions, task_type, must_refuse):
        if t != "safety":
            rewards.append(None)
            continue
        text = str(completion)
        if must:
            rewards.append(1.0 if any(tok in text for tok in refusal_markers) else 0.0)
        else:
            rewards.append(None)
    return rewards

reward_funcs = [
    math_reward_func,
    format_reward_func,
    persona_reward_func,
    safety_reward_func,
]

## Step 10. reward 함수 수동 테스트

실제 trainer 실행 전에 reward 함수가 원하는 방식으로 동작하는지 가볍게 확인합니다.

In [ ]:
sample_prompts = [row["prompt"] for row in eval_dataset.select(range(min(4, len(eval_dataset))))]
sample_task_type = [row.get("task_type") for row in eval_dataset.select(range(min(4, len(eval_dataset))))]
sample_gt = [row.get("ground_truth") for row in eval_dataset.select(range(min(4, len(eval_dataset))))]
sample_required_keys = [row.get("required_keys") for row in eval_dataset.select(range(min(4, len(eval_dataset))))]
sample_must_include = [row.get("must_include") for row in eval_dataset.select(range(min(4, len(eval_dataset))))]
sample_max_chars = [row.get("max_chars") for row in eval_dataset.select(range(min(4, len(eval_dataset))))]
sample_must_refuse = [row.get("must_refuse") for row in eval_dataset.select(range(min(4, len(eval_dataset))))]

sample_completions = [generate_text(model, tokenizer, p, max_new_tokens=64) for p in sample_prompts]

print("Sample completions:")
for p, c in zip(sample_prompts, sample_completions):
    print("=" * 80)
    print("PROMPT:", p)
    print("COMPLETION:", c)

print("\nReward test:")
print("math_reward_func   :", math_reward_func(sample_completions, task_type=sample_task_type, ground_truth=sample_gt))
print("format_reward_func :", format_reward_func(sample_completions, task_type=sample_task_type, required_keys=sample_required_keys))
print("persona_reward_func:", persona_reward_func(sample_completions, task_type=sample_task_type, must_include=sample_must_include, max_chars=sample_max_chars))
print("safety_reward_func :", safety_reward_func(sample_completions, task_type=sample_task_type, must_refuse=sample_must_refuse))

## Step 11. 학습 전 SFT 출력 저장

GRPO 학습 전의 출력을 저장해 두었다가, 학습 후 결과와 비교합니다.

In [ ]:
compare_rows = eval_dataset.select(range(min(8, len(eval_dataset))))

before_outputs = []
for row in compare_rows:
    out = generate_text(model, tokenizer, row["prompt"], max_new_tokens=64)
    before_outputs.append({
        "task_type": row.get("task_type"),
        "prompt": row["prompt"],
        "output_before_grpo": out
    })

before_path = os.path.join(OUTPUT_DIR, "module8_before_outputs.json")
with open(before_path, "w", encoding="utf-8") as f:
    json.dump(before_outputs, f, ensure_ascii=False, indent=2)

print("Saved:", before_path)

## Step 12. GRPOConfig 설정

Colab에서 무리 없이 돌리기 위한 교육용 기본값입니다.

핵심 포인트:
- `num_generations`: prompt당 몇 개 completion을 생성할지
- `beta=0.0`: 현재 TRL 문서 기준 기본 관행과 맞춤
- `max_prompt_length`, `max_completion_length`: 메모리 절약에 중요

> 메모리가 부족하면 `num_generations`를 2로 줄이거나 `max_completion_length`를 더 낮추세요.

In [ ]:
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_prompt_length=256,
    max_completion_length=64,
    num_generations=2,   # 메모리가 허용되면 4로 올릴 수 있음
    beta=0.0,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
)

## Step 13. GRPOTrainer 생성

이번 trainer는 다음을 사용합니다.

- 초기 모델: Module 4 SFT 결과
- reward 함수: math / format / persona / safety
- train dataset: `module3_grpo_dataset.jsonl`

In [ ]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_funcs,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
print(trainer)

## Step 14. 학습 실행 및 메모리/시간 측정

이번 셀에서는 학습 시간과 peak GPU memory를 함께 기록합니다.

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

peak_mem_gb = None
if torch.cuda.is_available():
    peak_mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(train_result)
print("Elapsed seconds:", round(elapsed, 2))
print("Peak GPU memory (GB):", None if peak_mem_gb is None else round(peak_mem_gb, 3))

## Step 15. 학습 결과 저장

학습된 GRPO 모델과 tokenizer를 저장합니다.

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

run_summary = {
    "initial_model_name": INITIAL_MODEL_NAME,
    "elapsed_seconds": elapsed,
    "peak_gpu_memory_gb": peak_mem_gb,
    "train_dataset_size": len(train_dataset),
    "eval_dataset_size": len(eval_dataset),
}

run_summary_path = os.path.join(OUTPUT_DIR, "module8_grpo_run_summary.json")
with open(run_summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

print("Saved:", run_summary_path)

## Step 16. 학습 후 GRPO 출력 생성

이제 같은 prompt에 대해 GRPO 학습 후 출력을 생성하고, 앞서 저장한 SFT 출력과 비교합니다.

In [ ]:
trained_model = trainer.model

after_outputs = []
for row in compare_rows:
    out = generate_text(trained_model, tokenizer, row["prompt"], max_new_tokens=64)
    after_outputs.append({
        "task_type": row.get("task_type"),
        "prompt": row["prompt"],
        "output_after_grpo": out
    })

after_path = os.path.join(OUTPUT_DIR, "module8_after_outputs.json")
with open(after_path, "w", encoding="utf-8") as f:
    json.dump(after_outputs, f, ensure_ascii=False, indent=2)

print("Saved:", after_path)

## Step 17. SFT vs GRPO 비교표 생성

같은 eval prompt에 대해 before/after를 합쳐서 저장합니다.

In [ ]:
comparison_rows = []
for b, a in zip(before_outputs, after_outputs):
    comparison_rows.append({
        "task_type": b["task_type"],
        "prompt": b["prompt"],
        "output_before_grpo": b["output_before_grpo"],
        "output_after_grpo": a["output_after_grpo"],
    })

comparison_path = os.path.join(OUTPUT_DIR, "module8_grpo_before_after.json")
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison_rows, f, ensure_ascii=False, indent=2)

print("Saved:", comparison_path)

for row in comparison_rows[:4]:
    print("=" * 100)
    print("TASK TYPE:", row["task_type"])
    print("PROMPT:", row["prompt"])
    print("BEFORE:", row["output_before_grpo"])
    print("AFTER :", row["output_after_grpo"])

## Step 18. 간단 scorecard 계산

훈련과 같은 reward 철학을 사용해, before/after 평균 보상을 간단히 비교합니다.

> 이 점수표는 **정식 benchmark**가 아니라 수업용 관찰 도구입니다.

In [ ]:
def aggregate_rewards(rows, output_key):
    completions = [r[output_key] for r in rows]
    task_type = [r["task_type"] for r in rows]

    task_meta = {r["prompt"]: r for r in compare_rows}
    ground_truth = [task_meta[r["prompt"]].get("ground_truth") for r in rows]
    required_keys = [task_meta[r["prompt"]].get("required_keys") for r in rows]
    must_include = [task_meta[r["prompt"]].get("must_include") for r in rows]
    max_chars = [task_meta[r["prompt"]].get("max_chars") for r in rows]
    must_refuse = [task_meta[r["prompt"]].get("must_refuse") for r in rows]

    reward_lists = [
        math_reward_func(completions, task_type=task_type, ground_truth=ground_truth),
        format_reward_func(completions, task_type=task_type, required_keys=required_keys),
        persona_reward_func(completions, task_type=task_type, must_include=must_include, max_chars=max_chars),
        safety_reward_func(completions, task_type=task_type, must_refuse=must_refuse),
    ]

    totals = []
    for i in range(len(rows)):
        vals = [rl[i] for rl in reward_lists if rl[i] is not None]
        totals.append(sum(vals) if vals else 0.0)

    return totals, reward_lists

before_totals, before_reward_lists = aggregate_rewards(comparison_rows, "output_before_grpo")
after_totals, after_reward_lists = aggregate_rewards(comparison_rows, "output_after_grpo")

scorecard = []
for idx, row in enumerate(comparison_rows):
    scorecard.append({
        "task_type": row["task_type"],
        "prompt": row["prompt"],
        "before_total_reward": before_totals[idx],
        "after_total_reward": after_totals[idx],
        "reward_delta": after_totals[idx] - before_totals[idx],
        "output_before_grpo": row["output_before_grpo"],
        "output_after_grpo": row["output_after_grpo"],
    })

scorecard_path = os.path.join(OUTPUT_DIR, "module8_grpo_scorecard.json")
with open(scorecard_path, "w", encoding="utf-8") as f:
    json.dump(scorecard, f, ensure_ascii=False, indent=2)

print("Saved:", scorecard_path)
print("Average reward before:", round(sum(before_totals) / len(before_totals), 4))
print("Average reward after :", round(sum(after_totals) / len(after_totals), 4))

## Step 19. 선택 사항: PPO 결과와 비교

만약 Module 7 notebook에서 생성한 요약 파일이 현재 세션에 있다면 함께 비교할 수 있습니다.  
없으면 이 셀은 그냥 건너뛰어도 됩니다.

In [ ]:
possible_ppo_files = [
    "/content/module7_ppo_output/module7_ppo_run_summary.json",
    "/content/module7_ppo_run_summary.json",
]

ppo_summary = None
for p in possible_ppo_files:
    if os.path.exists(p):
        with open(p, "r", encoding="utf-8") as f:
            ppo_summary = json.load(f)
        print("Loaded PPO summary from:", p)
        break

if ppo_summary is None:
    print("No PPO summary file found. Skip this comparison or upload your Module 7 summary manually.")
else:
    print(json.dumps(ppo_summary, ensure_ascii=False, indent=2))
    print("\nCurrent GRPO summary:")
    print(json.dumps(run_summary, ensure_ascii=False, indent=2))

## Step 20. 비교 메모 템플릿 생성

이 템플릿은 수강생이 PPO vs GRPO의 체감 차이를 정리하도록 돕습니다.

In [ ]:
summary_md = f"""# Module 8 GRPO Summary

## Run Summary
- initial_model_name: {INITIAL_MODEL_NAME}
- elapsed_seconds: {round(elapsed, 2)}
- peak_gpu_memory_gb: {None if peak_mem_gb is None else round(peak_mem_gb, 3)}
- train_dataset_size: {len(train_dataset)}
- eval_dataset_size: {len(eval_dataset)}

## Reward Summary
- average_reward_before: {round(sum(before_totals) / len(before_totals), 4)}
- average_reward_after: {round(sum(after_totals) / len(after_totals), 4)}

## SFT vs GRPO
- Which task improved the most?
- Which task changed little?
- Did GRPO help more on math / format / persona / safety?

## PPO vs GRPO Comparison Notes
- PPO elapsed_seconds:
- PPO peak_gpu_memory_gb:
- GRPO elapsed_seconds: {round(elapsed, 2)}
- GRPO peak_gpu_memory_gb: {None if peak_mem_gb is None else round(peak_mem_gb, 3)}

### Questions
1. Which felt more stable: PPO or GRPO?
2. Which felt simpler to implement?
3. Which looked lighter on GPU memory?
4. Which gave better output quality on this task mix?
5. In practice, which would you try first and why?
"""

summary_path = os.path.join(OUTPUT_DIR, "module8_grpo_summary.md")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_md)

print("Saved:", summary_path)
print(summary_md)

## Step 21. 결과 파일 확인

정상적으로 완료되면 아래 파일들이 생성됩니다.

- `module8_before_outputs.json`
- `module8_after_outputs.json`
- `module8_grpo_before_after.json`
- `module8_grpo_scorecard.json`
- `module8_grpo_run_summary.json`
- `module8_grpo_summary.md`

In [ ]:
print("Saved files:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print("-", os.path.join(OUTPUT_DIR, name))

## 선택 사항: 결과 다운로드

Colab 세션이 종료되기 전에 결과 파일을 다운로드할 수 있습니다.

In [ ]:
from google.colab import files

download_candidates = [
    os.path.join(OUTPUT_DIR, "module8_grpo_before_after.json"),
    os.path.join(OUTPUT_DIR, "module8_grpo_scorecard.json"),
    os.path.join(OUTPUT_DIR, "module8_grpo_run_summary.json"),
    os.path.join(OUTPUT_DIR, "module8_grpo_summary.md"),
]

for path in download_candidates:
    if os.path.exists(path):
        files.download(path)

# Module 8 정리

이번 모듈에서 우리는 다음을 경험했습니다.

- GRPO는 PPO와 마찬가지로 **online RL**
- 그러나 **group-relative advantage** 를 사용하고, DeepSeekMath/TRL 맥락에서는 **critic/value-model 의존성이 줄어든 경량 실험 구조**로 이해할 수 있음
- 같은 reward 철학을 써도 PPO와 GRPO는 **실행 복잡도, 로그 구조, 메모리 체감, 결과 양상**이 다르게 느껴질 수 있음

## 이번 모듈에서 말할 수 있어야 하는 문장

- **DPO는 offline preference alignment**
- **PPO는 reward/value 기반 online RL**
- **GRPO는 PPO 계열이지만 그룹 상대 비교를 활용해 더 가벼운 RL 실험을 설계할 수 있다**

## 다음 단계

이제 가장 자연스러운 마지막 단계는 다음입니다.

- SFT vs DPO vs PPO vs GRPO 통합 비교
- 과제 유형별 최적 학습법 정리
- 개인화 / 형식 준수 / 정답 최적화 / 추론 강화 문제를 각각 어떤 알고리즘으로 풀지 선택 전략 만들기